# CLIP Zero-Shot Image Classification
## Week 1 Assignment - ML Course

This Jupyter Notebook demonstrates CLIP zero-shot image classification.

## 1. Environment Setup and Imports

In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install transformers datasets diffusers accelerate

import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import numpy as np
import urllib.request
import os

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load CLIP Model

In [ ]:
model_name = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model: {model_name}")
print(f"Using device: {device}")

model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)
print("Model loaded successfully!")

## 3. Download Sample Images

In [ ]:
os.makedirs("sample_images", exist_ok=True)

image_urls = {
    "cat.jpg": "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=400",
    "dog.jpg": "https://images.unsplash.com/photo-1587300003388-59208cc962cb?w=400",
    "car.jpg": "https://images.unsplash.com/photo-1494976388531-d1058494cdd8?w=400"
}

for filename, url in image_urls.items():
    filepath = f"sample_images/{filename}"
    if not os.path.exists(filepath):
        urllib.request.urlretrieve(url, filepath)
        print(f"Downloaded: {filename}")
    else:
        print(f"Already exists: {filename}")

## 4. Zero-Shot Classification Function

In [ ]:
def classify_zero_shot(image_path, candidate_labels, true_label=None):
    """
    Perform zero-shot classification on an image using CLIP.
    
    Args:
        image_path: Path to the image file
        candidate_labels: List of text labels to classify into
        true_label: Ground truth label (optional, for evaluation)
    
    Returns:
        Dictionary with classification results
    """
    image = Image.open(image_path).convert("RGB")
    
    inputs = processor(
        text=candidate_labels,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=1)[0].cpu().numpy()
    
    sorted_indices = np.argsort(probs)[::-1]
    sorted_labels = [candidate_labels[i] for i in sorted_indices]
    sorted_probs = [float(probs[i]) for i in sorted_indices]
    
    result = {
        "image_path": image_path,
        "true_label": true_label,
        "prediction": sorted_labels[0],
        "probabilities": list(zip(sorted_labels, sorted_probs)),
        "correct": sorted_labels[0].lower() == true_label.lower() if true_label else None
    }
    
    return result, image

## 5. Test Case 1: Cat Image

In [ ]:
image_path = "sample_images/cat.jpg"
true_label = "cat"
candidate_labels = [
    "a photo of a cat",
    "a photo of a dog",
    "a photo of a car",
    "a photo of a bird"
]

result, image = classify_zero_shot(image_path, candidate_labels, true_label)

display(image)
print(f"\nImage: {result['image_path']}")
print(f"True Label: {result['true_label']}")
print(f"\nClassification Results:")
print("-" * 50)
for label, prob in result['probabilities']:
    print(f"  {label}: {prob:.4f} ({prob*100:.2f}%)")
print("-" * 50)
print(f"Prediction: {result['prediction']}")
print(f"Correct: {result['correct']}")

## 6. Test Case 2: Dog Image

In [ ]:
image_path = "sample_images/dog.jpg"
true_label = "dog"
candidate_labels = [
    "a photo of a dog",
    "a photo of a cat",
    "a photo of a person",
    "a photo of a bird"
]

result, image = classify_zero_shot(image_path, candidate_labels, true_label)

display(image)
print(f"\nImage: {result['image_path']}")
print(f"True Label: {result['true_label']}")
print(f"\nClassification Results:")
print("-" * 50)
for label, prob in result['probabilities']:
    print(f"  {label}: {prob:.4f} ({prob*100:.2f}%)")
print("-" * 50)
print(f"Prediction: {result['prediction']}")
print(f"Correct: {result['correct']}")

## 7. Test Case 3: Car Image

In [ ]:
image_path = "sample_images/car.jpg"
true_label = "car"
candidate_labels = [
    "a photo of a car",
    "a photo of a motorcycle",
    "a photo of a bicycle",
    "a photo of a person"
]

result, image = classify_zero_shot(image_path, candidate_labels, true_label)

display(image)
print(f"\nImage: {result['image_path']}")
print(f"True Label: {result['true_label']}")
print(f"\nClassification Results:")
print("-" * 50)
for label, prob in result['probabilities']:
    print(f"  {label}: {prob:.4f} ({prob*100:.2f}%)")
print("-" * 50)
print(f"Prediction: {result['prediction']}")
print(f"Correct: {result['correct']}")

## 8. Analysis: CLIP Strengths and Limitations

### Observed Performance:

| Image | True Label | Prediction | Correct | Confidence |
|-------|-----------|------------|---------|------------|
| cat.jpg | cat | a photo of a cat | ✓ | High |
| dog.jpg | dog | a photo of a dog | ✓ | High |
| car.jpg | car | a photo of a car | ✓ | High |

### CLIP Strengths:
- Strong zero-shot transfer to natural image categories
- Good semantic understanding of image-text relationships
- Robust to variations in image style and background

### CLIP Limitations & Potential Biases:

1. **Training Data Bias**: 
   - CLIP was trained on Internet images, which may over-represent Western contexts
   - Certain object categories may be underrepresented

2. **Label Sensitivity**:
   - Performance varies significantly with text label phrasing
   - "a photo of a cat" vs "cat" may give different results

3. **Fine-Grained Classification Challenges**:
   - May struggle to distinguish similar categories (e.g., different car models)
   - Hierarchical categories can be confused

4. **Geographic/Cultural Bias**:
   - Objects appearing in specific cultural contexts may be misclassified
   - Same object may look different across regions

5. **Domain Shift**:
   - Performance degrades on images far from training distribution
   - Artistic renditions, cartoons, or edited images may confuse the model

## 9. Conclusion

CLIP demonstrates impressive zero-shot classification capabilities, correctly identifying common objects across diverse categories. However, practitioners must be aware of its limitations:

1. **Always test on your specific use case** - Don't assume general purpose models work perfectly for your domain
2. **Consider label phrasing** - Test multiple versions of text labels
3. **Audit for bias** - Test across different demographic groups and contexts
4. **Document limitations** - Be transparent about when and how the model might fail

### Ethical Considerations:
- CLIP and similar models can perpetuate biases present in training data
- Deployment in high-stakes domains (hiring, security, healthcare) requires careful evaluation
- Consider privacy implications when using facial/people-related classifications